# Faz 1.1 — Kural Tabanlı (Motorlu) Baseline IFC Üretimi

**LLM yok.** Deterministik bir motorla (ifcopenshell) kurallara uygun bir bina
IFC'si üretiyoruz: 1 kat, 3 oda, giriş + iç kapılar, pencereler, döşeme, mekanlar.

**Tasarım:** Mantık `src/ifc_gen/baseline/` altında **fonksiyon** olarak yazılı;
bu notebook onları **import edip çağırıyor**. Aynı fonksiyonlar diğer fazlarda da
kullanılıyor (ör. Faz 3.1 ihlal ekleme `layout_to_ifc`'i yeniden kullanır).

| Fonksiyon | Ne yapar |
|-----------|----------|
| `medium_layout(...)` | Tasarım verisi (duvar/oda/kapı/pencere) üretir — IFC'den bağımsız |
| `layout_to_ifc(layout)` | Tasarımı geçerli IFC4 modeline çevirir (in-memory) |
| `build_baseline(layout)` | IFC üretir + `data/baseline_ifc/` altına yazar + `meta.json` |

## 0) Kurulum — `src/` yolunu ekle

In [ ]:
import sys, json
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = REPO / 'src'
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))
print('src:', SRC)

## 1) Tasarım verisi — `medium_layout()`
Önce IFC'den bağımsız, saf tasarım verisini üretiyoruz (metre).

In [ ]:
from ifc_gen.baseline import medium_layout

layout = medium_layout(width=12, depth=8, height=3, wall_thickness=0.2)
print('Parametreler:', layout.params)
print(f'Duvar: {len(layout.walls)} | Açıklık: {len(layout.openings)} | Oda: {len(layout.rooms)}')
print()
print('Duvarlar:')
for w in layout.walls:
    tip = 'dış' if w.is_external else 'iç'
    print(f'  {w.name:9s} {tip:3s}  uzunluk={w.length:.1f}m  kalınlık={w.thickness}m')
print('Açıklıklar:')
for o in layout.openings:
    print(f'  {o.name:12s} {o.kind:6s} @ {o.wall_name:8s} {o.width}×{o.height}m sill={o.sill}')

## 2) IFC üret — `build_baseline()`
Tasarımı geçerli IFC4'e çevirir, `data/baseline_ifc/` altına yazar, `meta.json` kaydeder.

In [ ]:
from ifc_gen.baseline import build_baseline

model, ifc_path, meta = build_baseline(layout)
print('Yazıldı:', ifc_path)
print('Şema:', meta['schema'], '| birim: metre')
print('Eleman sayıları:')
print(json.dumps(meta['counts'], indent=2))

## 3) Geçerlilik — şema + geometri
Üretilen IFC standart şemaya uygun mu, geometri çiziliyor mu?

In [ ]:
import ifcopenshell
from ifcopenshell import validate

f = ifcopenshell.open(str(ifc_path))
lg = validate.json_logger(); validate.validate(f, lg)
print('Şema hatası:', len(lg.statements))

import ifcopenshell.geom as geom
s = geom.settings(); it = geom.iterator(s, f)
n = bad = 0
if it.initialize():
    while True:
        if not it.get().geometry.verts: bad += 1
        n += 1
        if not it.next(): break
print(f'Geometri şekli: {n} | boş: {bad}')

## 4) Kural uyumu — baseline temiz olmalı
Kural tabanlı baseline **tüm kuralları sağlamalı** (ihlal sayısı = 0). Aynı kural seti Faz 3.1 ihlal eklemede de kullanılıyor.

In [ ]:
from ifc_gen.inject import rules
from viewer.model import load_viewer_model

vm = load_viewer_model(str(ifc_path))
ihlaller = rules.violations_only(vm)
print('Baseline ihlal sayısı (0 olmalı):', len(ihlaller))
for f in rules.detect(vm):
    durum = 'İHLAL' if f.violated else 'ok'
    print(f'  {f.rule:24s} {vm.elements[f.ekey].name:10s} {f.detail:30s} [{durum}]')

## 5) Görselleştir
Statik önizleme (her ortam) + tek satır interaktif `view(...)`.

In [ ]:
import matplotlib.pyplot as plt
from viewer import static
fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(121, projection='3d'); static.plot_3d(vm, ax=ax1)
ax2 = fig.add_subplot(122); static.plot_graph(vm, ax=ax2)
plt.tight_layout(); plt.show()

In [ ]:
from viewer import view
view(str(ifc_path))

## 6) Yeniden kullanım — farklı parametrelerle baseline
Fonksiyon olduğu için istediğin boyutta baseline üretebilirsin. Örnek: daha büyük bina.

In [ ]:
buyuk = medium_layout(width=15, depth=10, height=3.2)
_, buyuk_path, buyuk_meta = build_baseline(buyuk, filename='baseline_buyuk.ifc')
print('Yazıldı:', buyuk_path.name, '| sayılar:', buyuk_meta['counts'])
# view(str(buyuk_path))   # istersen görselleştir

## Özet
- Baseline mantığı **modüler fonksiyonlar** (`medium_layout` / `layout_to_ifc` / `build_baseline`).
- Çıktı: geçerli IFC4 (şema 0 hata, geometri tam) + `meta.json`, `data/baseline_ifc/` altında.
- Baseline **kurallara uygun** (0 ihlal) → ihlal ekleme (Faz 3) için temiz zemin.
- Sıradaki baseline yöntemleri: **1.2 tam LLM**, **1.3 hibrit (LLM + motor)**.